# MCP tool optimization POC

Proof of concept for **optimizing MCP (Model Context Protocol) tool metadata** using open-source patterns: an LLM rewrites tool descriptions and parameter schema text so an agent model can **select and call tools more reliably**.

- **Input:** MCP-style tool definitions (name, description, inputSchema).
- **Optimization:** Use an LLM (OpenAI-compatible API) to rewrite descriptions and parameter descriptions for agent-facing clarity.
- **Evaluation:** Small eval set of (user query, expected tool); prompt the same LLM "which tool?" with original vs optimized tool list and compare accuracy.

**References:** [mcp_optimization_ideas.md](../mcp_optimization_ideas.md) (Trace-Free+, PARSE, DSPy+MCP). This POC uses a simple LLM rewrite step rather than full Trace or DSPy.

## 1. Setup — LLM client and config

Use any **OpenAI-compatible** endpoint (e.g. RHOAI, OpenAI, local). Set `OPENAI_API_BASE` and `OPENAI_API_KEY` in env or below.

In [ ]:
import os
import json

# Optional: load from .env (e.g. OPENAI_API_KEY, OPENAI_API_BASE)
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

OPENAI_API_BASE = os.getenv("OPENAI_API_BASE", "https://api.openai.com/v1")  # or RHOAI inference URL + /v1
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")  # set for API calls
MODEL = os.getenv("MCP_POC_MODEL", "gpt-4o-mini")  # or llama-3.1-70b, etc.

def chat(messages: list[dict], temperature: float = 0.3) -> str:
    """Call OpenAI-compatible chat completions. Returns assistant content."""
    import httpx
    url = OPENAI_API_BASE.rstrip("/") + "/chat/completions"
    if "/v1" not in url and not url.endswith("completions"):
        url = OPENAI_API_BASE.rstrip("/") + "/v1/chat/completions"
    resp = httpx.post(
        url,
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}", "Content-Type": "application/json"},
        json={"model": MODEL, "messages": messages, "temperature": temperature},
        timeout=60.0,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print("Config: model =", MODEL, ", base =", OPENAI_API_BASE)

## 2. Sample MCP-style tools

In a real flow these would come from `tools/list` on an MCP server. Here we define a few tools with **short or vague** descriptions so we can show improvement after optimization.

In [ ]:
TOOLS_ORIGINAL = [
    {
        "name": "fetch_weather",
        "description": "Gets the weather for a location.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "Location"},
                "unit": {"type": "string", "description": "Unit"},
            },
            "required": ["location"],
        },
    },
    {
        "name": "search_docs",
        "description": "Search documentation.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Query"},
            },
            "required": ["query"],
        },
    },
    {
        "name": "run_query",
        "description": "Execute a query.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "sql": {"type": "string", "description": "SQL"},
            },
            "required": ["sql"],
        },
    },
]

def format_tools_for_prompt(tools: list[dict]) -> str:
    """Format tool list for inclusion in an LLM prompt."""
    parts = []
    for t in tools:
        parts.append(f"- **{t['name']}**: {t['description']}")
        props = t.get("inputSchema", {}).get("properties", {})
        if props:
            for k, v in props.items():
                desc = v.get("description", "") or "(no description)"
                parts.append(f"  - {k}: {desc}")
    return "\n".join(parts)

print("Original tools:")
print(format_tools_for_prompt(TOOLS_ORIGINAL))

## 3. Optimize tool descriptions with LLM

Prompt the LLM to rewrite each tool's **description** and **parameter descriptions** so an agent can better decide when to use the tool and how to fill arguments.

In [ ]:
def optimize_tool_description(tool: dict) -> dict:
    """Use LLM to rewrite tool description and parameter descriptions. Returns a new tool dict."""
    name = tool["name"]
    desc = tool["description"]
    schema = json.loads(json.dumps(tool.get("inputSchema", {})))
    props = schema.get("properties", {})

    prompt = f"""You are improving tool metadata for an LLM agent that must choose which tool to call.
Rewrite the following tool so the agent can reliably decide when to use it and how to fill parameters.
Output valid JSON only, with two keys: "description" (string) and "parameter_descriptions" (object mapping each parameter name to a short description).

Tool name: {name}
Current description: {desc}
Current parameters: {json.dumps(list(props.keys()))}

Output JSON:"""

    if not OPENAI_API_KEY:
        # No API: return a hand-crafted improvement for demo
        improved = {
            "description": "Returns current weather for a given location. Use when the user asks about weather, temperature, or conditions in a city/region. Input: location (string), optional unit (celsius|fahrenheit).",
            "parameter_descriptions": {"location": "City or region, e.g. San Francisco", "unit": "celsius or fahrenheit"},
        }
        if name == "search_docs":
            improved = {"description": "Search documentation by natural language query. Use when the user asks to find docs, look up how-to, or search help.", "parameter_descriptions": {"query": "Search query in natural language"}}
        elif name == "run_query":
            improved = {"description": "Execute a read-only SQL query against the database. Use when the user asks for data, records, or analytics. Do not use for weather or docs.", "parameter_descriptions": {"sql": "Valid SQL SELECT statement"}}
    else:
        out = chat([{"role": "user", "content": prompt}])
        out = out.strip().replace("```json", "").replace("```", "").strip()
        improved = json.loads(out)

    new_tool = {"name": name, "description": improved["description"], "inputSchema": schema}
    for param, param_desc in improved.get("parameter_descriptions", {}).items():
        if param in new_tool["inputSchema"].get("properties", {}):
            new_tool["inputSchema"]["properties"][param]["description"] = param_desc
    return new_tool

TOOLS_OPTIMIZED = [optimize_tool_description(t) for t in TOOLS_ORIGINAL]
print("Optimized tools:")
print(format_tools_for_prompt(TOOLS_OPTIMIZED))

## 4. Evaluation — tool selection accuracy

Eval set: (user query, expected tool name). We ask the LLM "which single tool should be used for this query?" with either the original or optimized tool list and parse the tool name. Compare accuracy before vs after.

In [ ]:
EVAL_SET = [
    ("What's the weather in Berlin?", "fetch_weather"),
    ("Temperature in Tokyo in Celsius", "fetch_weather"),
    ("Find docs on how to install", "search_docs"),
    ("Search for API authentication", "search_docs"),
    ("How many users signed up last week?", "run_query"),
    ("SELECT * FROM orders LIMIT 10", "run_query"),
]

def run_tool_selection_eval(tools: list[dict], eval_set: list[tuple]) -> float:
    """Prompt LLM to pick one tool for each query; return accuracy (0-1)."""
    tool_list = format_tools_for_prompt(tools)
    correct = 0
    for query, expected in eval_set:
        prompt = f"""Given these tools:
{tool_list}

User query: "{query}"
Which single tool should be used? Reply with only the tool name, nothing else."""
        if OPENAI_API_KEY:
            reply = chat([{"role": "user", "content": prompt}]).strip().lower()
        else:
            reply = expected  # demo: assume correct when no API
        # Normalize: allow "fetch_weather" or "Fetch_weather"
        if expected.lower() in reply or reply in expected.lower():
            correct += 1
    return correct / len(eval_set) if eval_set else 0.0

acc_orig = run_tool_selection_eval(TOOLS_ORIGINAL, EVAL_SET)
acc_opt  = run_tool_selection_eval(TOOLS_OPTIMIZED, EVAL_SET)
print(f"Eval set size: {len(EVAL_SET)}")
print(f"Tool selection accuracy (original): {acc_orig:.0%}")
print(f"Tool selection accuracy (optimized): {acc_opt:.0%}")
print(f"Change: {(acc_opt - acc_orig):+.0%}")

## 6. Open-source extensions and related publications

This section maps **open-source projects** from [mcp_optimization_ideas.md](../mcp_optimization_ideas.md) and **arXiv/publications** on tool description optimization, schema optimization, and LLM agent tool use. Use the code below to explore extensions (e.g. DSPy+MCP if installed) and to print a curated list of papers.

In [ ]:
# --- Open-source extensions (from mcp_optimization_ideas.md) ---
OPEN_SOURCE_EXTENSIONS = [
    {
        "name": "Microsoft Trace",
        "url": "https://github.com/microsoft/Trace",
        "role": "Tool descriptions + schema rewriting",
        "note": "Trace-Free+: curriculum learning to rewrite tool descriptions and parameter schemas for reliable agent tool use (trace-rich → trace-free). arXiv:2602.20426.",
    },
    {
        "name": "DSPy + MCP",
        "url": "https://dspy.ai/learn/programming/mcp",
        "role": "Tool-usage optimization",
        "note": "Convert MCP tools to DSPy tools; compile ReAct (etc.) with BootstrapFewShot/MIPRO to optimize when and how the model calls tools. Complements metadata rewriting.",
    },
    {
        "name": "GreaterPrompt",
        "url": "https://github.com/psunlpgroup/GreaterPrompt",
        "role": "Prompt optimization",
        "note": "Optimize the prompt text that describes or lists tools (system message). Multiple methods, Web UI.",
    },
    {
        "name": "PromptPerfect",
        "url": "https://github.com/Beagle-AI-automation/promptperfect",
        "role": "Prompt improvement",
        "note": "Improve prose that explains when/how to use tools (e.g. Make it Better, Add Chain-of-Thought).",
    },
    {
        "name": "PARSE (ARCHITECT)",
        "url": "https://arxiv.org/abs/2510.08623",
        "role": "Schema optimization",
        "note": "LLM-driven JSON schema optimization for LLM consumption; backward compatibility via RELAY. Entity extraction focus; idea applies to MCP inputSchema.",
    },
    {
        "name": "BentoML llm-optimizer",
        "url": "https://github.com/bentoml/llm-optimizer",
        "role": "Inference tuning",
        "note": "Benchmark and tune inference (vLLM, SGLang). For the model that calls MCP tools (latency/throughput), not tool text.",
    },
]

print("Open-source extensions for MCP tool optimization:\n")
for ext in OPEN_SOURCE_EXTENSIONS:
    print(f"  • {ext['name']} — {ext['role']}")
    print(f"    {ext['url']}")
    print(f"    {ext['note']}\n")

In [ ]:
# --- Optional: explore DSPy + MCP (if dspy[mcp] installed) ---
try:
    import dspy
    from dspy.tool import Tool
    # DSPy can convert MCP tools: Tool.from_mcp_tool(mcp_tool) after connecting to MCP server
    print("DSPy is installed. To use MCP: pip install 'dspy-ai[mcp]'; connect to MCP server, list tools, then Tool.from_mcp_tool(...) for each.")
    if hasattr(Tool, "from_mcp_tool"):
        print("  Tool.from_mcp_tool is available.")
    else:
        print("  (Tool.from_mcp_tool may be in dspy.tool or mcp module; see dspy.ai/learn/programming/mcp)")
except ImportError as e:
    print("DSPy not installed. To explore DSPy+MCP: pip install dspy-ai 'dspy-ai[mcp]'")
    print("  Then use MCP tools inside a ReAct program and compile with BootstrapFewShot/MIPRO.")

In [ ]:
# --- arXiv and related publications (tool description, schema, agent tool use) ---
ARXIV_PUBLICATIONS = [
    {
        "arxiv_id": "2602.20426",
        "title": "Learning to Rewrite Tool Descriptions for Reliable LLM-Agent Tool Use",
        "summary": "Trace-Free+: curriculum learning to rewrite tool descriptions and parameter schemas; transfer from trace-rich to trace-free; scales to 100+ tools.",
        "url": "https://arxiv.org/abs/2602.20426",
    },
    {
        "arxiv_id": "2510.08623",
        "title": "PARSE: LLM Driven Schema Optimization for Reliable Entity Extraction",
        "summary": "ARCHITECT optimizes JSON schemas for LLM consumption with backward compatibility (RELAY); SCOPE for reflection-based extraction. Up to 64.7% improvement on SWDE.",
        "url": "https://arxiv.org/abs/2510.08623",
    },
    {
        "arxiv_id": "2510.05592",
        "title": "In-the-Flow Agentic System Optimization for Effective Planning and Tool Use",
        "summary": "AgentFlow: Flow-GRPO for modular agentic systems; coordinates specialized modules; 14.9–14.5% accuracy gains, better tool-calling reliability.",
        "url": "https://arxiv.org/abs/2510.05592",
    },
    {
        "arxiv_id": "2510.00023",
        "title": "ToolBrain: A Flexible Reinforcement Learning Framework for Agentic Tools",
        "summary": "RL framework (GRPO, DPO, LLM-as-judge rewards) for tool use; up to 30% improvement in tool-use performance.",
        "url": "https://arxiv.org/abs/2510.00023",
    },
    {
        "arxiv_id": "2512.02840",
        "title": "promptolution: A Unified, Modular Framework for Prompt Optimization",
        "summary": "Modular prompt optimization with token budgets; apply to tool-use instructions or tool-summary prompt.",
        "url": "https://arxiv.org/abs/2512.02840",
    },
    {
        "arxiv_id": "2511.14650",
        "title": "AutoTool: Efficient Tool Selection for Large Language Model Agents",
        "summary": "Graph-based tool selection exploiting tool usage inertia; up to 30% inference cost reduction.",
        "url": "https://arxiv.org/abs/2511.14650",
    },
    {
        "arxiv_id": "2411.04535",
        "title": "Meta-Reasoning Improves Tool Use in Large Language Models",
        "summary": "TECTON: two-phase (candidate tools + meta-reasoning for final selection); gains on math reasoning.",
        "url": "https://arxiv.org/abs/2411.04535",
    },
]

print("arXiv and related publications (tool description, schema, agent tool use):\n")
for p in ARXIV_PUBLICATIONS:
    print(f"  [{p['arxiv_id']}] {p['title']}")
    print(f"      {p['summary']}")
    print(f"      {p['url']}\n")

### 6.1 POC: Using extensions and paper algorithms

The cells below implement **minimal POC code** that applies ideas from Section 6 to the MCP tool optimization problem:

- **Trace-Free+ style** (arXiv:2602.20426): Curriculum-style rewrite — first "trace-rich" (with example queries), then "trace-free" description.
- **PARSE/ARCHITECT style** (arXiv:2510.08623): LLM-driven optimization of `inputSchema` for clearer LLM consumption.
- **promptolution-style** (arXiv:2512.02840): Optimize the tool-list prompt under a token/word budget.
- **TECTON-style** (arXiv:2411.04535): Two-phase tool selection — candidate retrieval then meta-reasoning for final choice.
- **DSPy + MCP**: Convert MCP-style tool dicts to DSPy tools and run a ReAct program (with optional BootstrapFewShot compile).

In [ ]:
# --- POC: Trace-Free+ style (curriculum rewrite) + PARSE-style (schema optimization) ---

def rewrite_tool_trace_free_plus_style(tool: dict, example_queries: list[str] | None = None) -> dict:
    """
    Trace-Free+ style (arXiv:2602.20426): curriculum learning.
    Step 1 (trace-rich): optionally include example queries that use this tool.
    Step 2 (trace-free): rewrite description so the agent can use it without traces.
    """
    name, desc = tool["name"], tool["description"]
    schema = json.loads(json.dumps(tool.get("inputSchema", {})))
    props = schema.get("properties", {})
    examples = example_queries or [f"Example user request that should use {name}"]
    trace_rich = f"{desc}\nExample queries that use this tool: " + "; ".join(examples[:3])
    prompt = f"""Rewrite this tool description to be "trace-free": clear and self-contained so an LLM agent can decide when to use the tool and how to fill parameters, WITHOUT needing the example queries.
Tool name: {name}
Trace-rich description: {trace_rich}
Parameters: {list(props.keys())}
Output JSON with keys: "description" (string), "parameter_descriptions" (object). Output JSON only."""
    if not OPENAI_API_KEY:
        return optimize_tool_description(tool)  # fallback to Section 3
    out = chat([{"role": "user", "content": prompt}]).strip().replace("```json", "").replace("```", "").strip()
    improved = json.loads(out)
    new_tool = {"name": name, "description": improved["description"], "inputSchema": schema}
    for p, pd in improved.get("parameter_descriptions", {}).items():
        if p in new_tool["inputSchema"].get("properties", {}):
            new_tool["inputSchema"]["properties"][p]["description"] = pd
    return new_tool

def optimize_schema_parse_style(tool: dict) -> dict:
    """
    PARSE/ARCHITECT style (arXiv:2510.08623): LLM-driven schema optimization for LLM consumption.
    Returns tool with optimized inputSchema (clearer property names/descriptions, simplified for the model).
    """
    name, desc = tool["name"], tool["description"]
    schema = tool.get("inputSchema", {})
    prompt = f"""Optimize this JSON schema for an LLM that will choose and call the tool. Keep the same property names and types. Improve only the descriptions to be clear and actionable. Output a single JSON object for the full inputSchema (type, properties, required).
Tool: {name}. Description: {desc}
Current inputSchema: {json.dumps(schema)}
Output the optimized inputSchema JSON only."""
    if not OPENAI_API_KEY:
        return tool
    out = chat([{"role": "user", "content": prompt}]).strip().replace("```json", "").replace("```", "").strip()
    try:
        new_schema = json.loads(out)
        return {"name": name, "description": desc, "inputSchema": new_schema}
    except json.JSONDecodeError:
        return tool

# Demo: apply Trace-Free+ to one tool (with example queries)
example_queries = ["What's the weather in Berlin?", "Temperature in Tokyo in Celsius"]
tool_trace_free = rewrite_tool_trace_free_plus_style(TOOLS_ORIGINAL[0], example_queries)
print("Trace-Free+ style (fetch_weather):", tool_trace_free["description"][:120] + "...")

# Demo: PARSE-style schema optimization (if API key set)
tool_parse = optimize_schema_parse_style(TOOLS_ORIGINAL[1])
print("PARSE-style schema (search_docs) properties:", list(tool_parse.get("inputSchema", {}).get("properties", {}).keys()))

In [ ]:
# --- POC: promptolution-style (arXiv:2512.02840) — tool-list prompt under token/word budget ---

def optimize_tool_list_prompt_budget(tools: list[dict], max_words: int = 80) -> str:
    """
    Optimize the tool-list prompt under a word budget (promptolution-style).
    Returns a shortened, agent-friendly summary of tools that fits the budget.
    """
    full_list = format_tools_for_prompt(tools)
    prompt = f"""Summarize the following tool list for an LLM agent that must choose one tool per user query. Use at most {max_words} words. Be clear and preserve tool names and when to use each.

Tools:
{full_list}

Short summary (at most {max_words} words):"""
    if not OPENAI_API_KEY:
        # Fallback: truncate by tool count
        lines = full_list.split("\n")
        return "\n".join(lines[: min(len(lines), 6)]) + ("..." if len(lines) > 6 else "")
    return chat([{"role": "user", "content": prompt}]).strip()

budget_prompt = optimize_tool_list_prompt_budget(TOOLS_OPTIMIZED, max_words=60)
print("Tool list under 60-word budget:")
print(budget_prompt)

In [ ]:
# --- POC: TECTON-style (arXiv:2411.04535) — two-phase tool selection: candidates + meta-reasoning ---

def get_tool_candidates(query: str, tools: list[dict], top_k: int = 2) -> list[dict]:
    """Phase 1: Retrieve candidate tools (e.g. by keyword overlap)."""
    q_lower = set(query.lower().split())
    scored = []
    for t in tools:
        name = t.get("name", "")
        desc = (t.get("description") or "").lower()
        words = set((name + " " + desc).split())
        overlap = len(q_lower & words)
        scored.append((overlap, t))
    scored.sort(key=lambda x: -x[0])
    return [t for _, t in scored[:top_k]]

def select_tool_tecton_style(query: str, tools: list[dict], top_k: int = 2) -> str | None:
    """
    Two-phase: (1) get candidate tools, (2) meta-reasoning LLM call to pick one.
    Returns tool name or None.
    """
    candidates = get_tool_candidates(query, tools, top_k=top_k)
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]["name"]
    tool_list = format_tools_for_prompt(candidates)
    prompt = f"""Among these candidate tools only, which single tool should be used for this user query? Reply with only the tool name.

Candidates:
{tool_list}

User query: "{query}"
Tool name:"""
    if not OPENAI_API_KEY:
        return candidates[0]["name"]
    reply = chat([{"role": "user", "content": prompt}]).strip().lower()
    for t in candidates:
        if t["name"].lower() in reply or reply in t["name"].lower():
            return t["name"]
    return candidates[0]["name"]

# Eval TECTON-style on EVAL_SET
tecton_correct = sum(1 for q, exp in EVAL_SET if select_tool_tecton_style(q, TOOLS_OPTIMIZED, top_k=2) == exp)
print(f"TECTON-style two-phase tool selection accuracy (top_k=2): {tecton_correct}/{len(EVAL_SET)} = {tecton_correct/len(EVAL_SET):.0%}")

In [ ]:
# --- POC: DSPy + MCP — convert MCP-style tools to DSPy ReAct and run (optional compile) ---

def mcp_style_tools_to_dspy_tools(tools: list[dict]):
    """
    Convert MCP-style tool dicts to DSPy-callable tools (functions with docstrings).
    In production, use dspy Tool.from_mcp_tool() when connected to an MCP server (pip install 'dspy-ai[mcp]').
    """
    def make_tool(t):
        name, desc = t["name"], t.get("description", "")
        params = t.get("inputSchema", {}).get("properties", {})
        def fn(**kwargs):
            return f"[called {name} with {kwargs}]"
        fn.__name__ = name
        fn.__doc__ = desc + "\nParams: " + ", ".join(params.keys())
        return fn
    return [make_tool(t) for t in tools]

try:
    import dspy
    from dspy.teleprompt import BootstrapFewShot
    dspy_tools = mcp_style_tools_to_dspy_tools(TOOLS_OPTIMIZED)
    react_agent = dspy.ReAct(
        signature="question -> answer",
        tools=dspy_tools,
        max_iters=3,
    )
    # Configure LM if we have API (OpenAI-compatible)
    if OPENAI_API_KEY:
        dspy.configure(lm=dspy.LM(f"openai/{MODEL}", api_key=OPENAI_API_KEY, api_base=OPENAI_API_BASE.rstrip("/")))
    # Single prediction
    pred = react_agent(question="What's the weather in Paris?")
    print("DSPy ReAct prediction:", getattr(pred, "answer", str(pred)))
    # Optional: compile with BootstrapFewShot on EVAL_SET (needs dspy.Example)
    # trainset = [dspy.Example(question=q, answer=exp).with_inputs("question") for q, exp in EVAL_SET]
    # optimizer = BootstrapFewShot(metric=lambda ex, pred, trace: 1.0 if pred.answer == ex.answer else 0.0)
    # compiled = optimizer.compile(react_agent, trainset=trainset)
    print("DSPy ReAct POC: agent ran successfully. To optimize, uncomment BootstrapFewShot and compile.")
except ImportError as e:
    print("DSPy not installed. To run this POC: pip install dspy-ai")
    print("For MCP server integration: pip install 'dspy-ai[mcp]' and use Tool.from_mcp_tool() from the MCP server tools list.")

## 7. Summary

- **POC:** LLM-based rewrite of MCP tool descriptions and parameter descriptions; then eval with a small (query, expected_tool) set to measure tool selection accuracy.
- **Section 6** lists **open-source extensions** (Trace, DSPy+MCP, GreaterPrompt, PromptPerfect, PARSE, BentoML llm-optimizer) from [mcp_optimization_ideas.md](../mcp_optimization_ideas.md) and a curated set of **arXiv/publications** on tool description rewriting, schema optimization, and LLM agent tool use.
- **Section 6.1** adds **POC source code** that uses those ideas: Trace-Free+ style curriculum rewrite, PARSE-style schema optimization, promptolution-style token-budget tool summary, TECTON-style two-phase tool selection, and DSPy ReAct with MCP-style tools (optional BootstrapFewShot compile). Run the cells to reproduce or extend.